# Qwen3.5 ハンズオン: vLLM + GPU で画像 + テキスト推論

このNotebookは **`Qwen/Qwen3.5-2B` を vLLM 上で GPU 推論** する最小ハンズオンです。

このリポジトリでは、arm64 + CUDA 13 + GB10 環境で vLLM を動かすために、**専用の `.venv-vllm`** を使っています。

## 0. 前提

- モデル: `Qwen/Qwen3.5-2B`
- 方式: **vLLM offline inference**
- モダリティ: **画像 + テキスト**
- 実行デバイス: **GPU (`cuda`)**

> 重要: 今回の環境では vLLM を source build しているため、Notebook本体の kernel ではなく `!` 付きシェル実行で `.venv-vllm` を使う構成にしています。

In [ ]:
!nvidia-smi
import os
print('workspace ok:', os.path.exists('../scripts/test_vllm_qwen35_mm_offline.py'))

## 1. サンプル画像を確認

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt

image_path = Path('../data/qwen35_demo.png')
image = Image.open(image_path).convert('RGB')
plt.figure(figsize=(8, 5))
plt.imshow(image)
plt.axis('off')
plt.show()
print(image_path)

## 2. vLLM で実行

下のスクリプトは内部で:
- `AutoProcessor.from_pretrained('Qwen/Qwen3.5-2B')`
- vLLM `LLM(...)`
- `multi_modal_data={'image': image}`

を使って、**画像 + テキスト入力**を GPU で実行します。

In [ ]:
!/project/.venv-vllm/bin/python ../scripts/test_vllm_qwen35_mm_offline.py

## 3. 理解ポイント

Qwen3.5 は `-VL` ではなくても **そのままマルチモーダル** です。
vLLM では Hugging Face の chat template を使って prompt を作り、`multi_modal_data` に画像を渡します。

基本形は次の通りです。

```python
processor = AutoProcessor.from_pretrained('Qwen/Qwen3.5-2B')
prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

outputs = llm.generate(
    {
        'prompt': prompt,
        'multi_modal_data': {'image': image},
    },
    sampling_params=SamplingParams(max_tokens=128, temperature=0.0),
)
```